# Protocol B

**Trạng thái:** `FROZEN_CONFIG`  
**Chế độ mặc định:** CPU · read-only · không train · không resume · không chạy test inference.  

Indian Pines · 16 lớp · 11×11 patch · PCA 40 · protocol `50e988fae260`. Các kiểm tra dưới đây chỉ đọc file đầu vào đã có.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'manifest.json').is_file() and (ROOT.parent / 'manifest.json').is_file():
    ROOT = ROOT.parent
elif not (ROOT / 'manifest.json').is_file() and (ROOT / 'KLTN_clean' / 'manifest.json').is_file():
    ROOT = ROOT / 'KLTN_clean'
if not (ROOT / 'manifest.json').is_file():
    raise FileNotFoundError('Project root not found. Open the extracted project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.audit import (verify_sources, scan, zip_listing, zip_job_contracts,
                       read_test_summary, read_test_by_seed, npz_headers,
                       source_functions, function_source, statuses, export_json)

# Optional directories containing original ZIPs and data.
import json
from src.audit import resolve_inputs
INPUT_CONFIG = resolve_inputs(ROOT)
INPUTS = INPUT_CONFIG['directories']
from src.results import (read_csv_file_or_zip, metrics_by_seed,
                         test_paired_deltas, recompute_summary,
                         check_against_stored, weighting_tables)
print('Project:', ROOT)

Project: C:\Users\ADMIN\OneDrive\Tài liệu\ChatGPT\KLTN\KLTN_clean


## Source lock

In [2]:
checks = verify_sources(ROOT)
print('Frozen source files:', len(checks), '| PASS:', sum(x['status']=='PASS' for x in checks))

Frozen source files: 15 | PASS: 15


## Dữ liệu

Khai báo thư mục ngoài trong `INPUTS` khi sử dụng cache B. Không tự refit PCA/scaler.

In [3]:
from src.audit import PROTOCOL_ID
CACHE = INPUT_CONFIG['protocol_b']  # Original B ZIP or directory
candidates = scan(([ROOT / 'data'] + INPUTS), extensions=('.zip','.mat','.npz','.json'))
print('Protocol:', PROTOCOL_ID, '| candidate files:', len(candidates))
for p in candidates[:25]: print(p.name)

Protocol: 50e988fae260 | candidate files: 9
01_PROTOCOL_B_SOURCE.zip
01_PROTOCOL_B_RESULTS.zip
02_M4_TRAINING_RESULTS.zip
03_M4_TEST_RESULTS.zip
04_V2_CLASSIFICATION_DIAGNOSTICS.zip
05_V2_WEIGHTING_8000.zip
06_ENDGAME_7500_8000_STATUS.zip
MOC2_IP_SMOKE_20261015T133956Z_2abba22f_REPORT.zip
V2_IP_SMOKE_20261015T120421Z_b948803d_REPORT.zip


## Cấu trúc cache

Chỉ đọc danh sách file ZIP/NPZ; không nạp toàn bộ tensors hoặc tạo lại split.

In [4]:
from pathlib import Path
if CACHE:
    source=Path(CACHE)
    if source.is_file() and source.suffix.lower()=='.zip':
        members=zip_listing(source)
        print('ZIP members:',len(members))
        for name in members[:30]:print(name)
    elif source.is_dir():
        for p in sorted(source.rglob('train_5528.npz')):
            print('Train cache:',p,'| schemas:',npz_headers(p))
    else:raise FileNotFoundError(source)
else:print('SKIP: cache B chưa được chỉ định')

ZIP members: 475
KLTN_HyperGAMO_B_50e988fae260/BAO_CAO_HINH.html
KLTN_HyperGAMO_B_50e988fae260/B_master_split.npz
KLTN_HyperGAMO_B_50e988fae260/B_master_split.npz.json
KLTN_HyperGAMO_B_50e988fae260/cnn_frozen_config.json
KLTN_HyperGAMO_B_50e988fae260/cpu_preflight_pass.json
KLTN_HyperGAMO_B_50e988fae260/development/dev_split.npz
KLTN_HyperGAMO_B_50e988fae260/development/dev_split.npz.json
KLTN_HyperGAMO_B_50e988fae260/development/dev_train_val.npz
KLTN_HyperGAMO_B_50e988fae260/development/dev_train_val.npz.json
KLTN_HyperGAMO_B_50e988fae260/development/overlap_audit.json
KLTN_HyperGAMO_B_50e988fae260/development/preprocessing_dev.joblib
KLTN_HyperGAMO_B_50e988fae260/development/preprocessing_dev.joblib.json
KLTN_HyperGAMO_B_50e988fae260/development/tuning/class_weight/best.json
KLTN_HyperGAMO_B_50e988fae260/development/tuning/class_weight/done.json
KLTN_HyperGAMO_B_50e988fae260/development/tuning/class_weight/lr_1e-03/best.weights.h5
KLTN_HyperGAMO_B_50e988fae260/development/tuning/cla

## Ranh giới protocol

Không dùng kết quả test để fit preprocessing. Báo cáo overlap không gian theo kiểm chứng lịch sử.

In [5]:
print('Split cuối theo hồ sơ: train 5,528; test 4,721; development 4,422/1,106')
print('Spatial independence: NOT established (patch footprints overlap)')
print('Source: docs/protocol_review.md and archive/legacy/KLTN_KiemChung_RoRiKhongGian_ReviewNhom.ipynb')

Split cuối theo hồ sơ: train 5,528; test 4,721; development 4,422/1,106
Spatial independence: NOT established (patch footprints overlap)
Source: docs/protocol_review.md and archive/legacy/KLTN_KiemChung_RoRiKhongGian_ReviewNhom.ipynb


## Giới hạn

Không thay đổi source huấn luyện đã chạy, không mở test mới và không kết luận PASS khi thiếu output gốc. Kết quả chỉ có giá trị trong phạm vi protocol B của Indian Pines.

In [6]:
import sys
assert 'tensorflow' not in sys.modules, 'Unexpected TensorFlow/model initialization'
print('CPU_ONLY_NO_MODEL_IMPORT_PASS')

CPU_ONLY_NO_MODEL_IMPORT_PASS
